# HNSW + 在线 HMM 检索与 Recall@25 评估

使用真实 NetVLAD 数据：多轨迹 Query，每条轨迹按帧做 HNSW Top-K 检索后经 OnlineHMM 时序平滑；
GT 采用坐标法（uav_infos.csv 经纬度 + 方圆 25 张），评估 Recall@25，对比「仅 HNSW」与「HNSW+HMM」。

In [1]:
import sys
import importlib
from pathlib import Path

_root = Path().resolve()
if _root.name == "hnsw_performance_analysis":
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import faiss

from demo_readH5 import (
    load_netvlad_descriptors,
    load_multi_netvlad_descriptors,
    DB_H5_PATHS,
    QUERY_H5_PATHS,
)
from scene_config import (
    get_db_scene_ranges,
    get_query_trajectory_ranges,
    get_scene_paths,
    DATASETS_BASE,
)
import coords_gt_utils
importlib.reload(coords_gt_utils)
from coords_gt_utils import (
    build_database_coords,
    build_gt_9_for_all_trajectories,
    get_geotransform_and_srs,
    lonlat_to_pixel,
    load_uav_infos,
)
from HMM.HMM import OnlineHMM

## 1. 加载 DB 与 Query（多轨迹）

In [2]:
print("加载 DB...")
db_names, db_descs = load_multi_netvlad_descriptors(DB_H5_PATHS)
db_descs = db_descs.astype(np.float32)
N_db, D = db_descs.shape
print(f"DB: {N_db} 条, 维度 {D}")

print("按轨迹加载 Query...")
query_per_traj = []
for h5_path in QUERY_H5_PATHS:
    names, descs = load_netvlad_descriptors(Path(h5_path))
    query_per_traj.append((names, descs.astype(np.float32)))

query_names = []
query_descs_list = []
for _, (names, descs) in enumerate(query_per_traj):
    query_names.extend(names)
    query_descs_list.append(descs)
query_descs = np.vstack(query_descs_list)
trajectory_ranges = get_query_trajectory_ranges(QUERY_H5_PATHS)
n_queries = query_descs.shape[0]
print(f"Query: {n_queries} 条, 共 {len(trajectory_ranges)} 条轨迹")
db_scene_ranges = get_db_scene_ranges(DB_H5_PATHS, db_names)

加载 DB...
DB: 11009 条, 维度 4096
按轨迹加载 Query...
Query: 5988 条, 共 15 条轨迹


## 2. DB 坐标（供 HMM 转移约束）与坐标法 GT@25

In [3]:
print("构建 database_coords（GDAL + id_startx_starty）...")
database_coords = build_database_coords(db_names, db_scene_ranges, DATASETS_BASE)
n_valid = np.sum(~np.any(np.isnan(database_coords), axis=1))
print(f"有效坐标: {n_valid}/{N_db}")

print("构建坐标法 GT@25（每 query 方圆 25 张）...")
gt_9_list, gt_25_ordered = build_gt_9_for_all_trajectories(
    trajectory_ranges,
    db_scene_ranges,
    db_names,
    DATASETS_BASE,
)
n_with_gt = sum(1 for s in gt_9_list if len(s) > 0)
print(f"有 GT 的 query 数: {n_with_gt}/{len(gt_9_list)}")
if n_with_gt > 0:
    sizes = [len(s) for s in gt_9_list if len(s) > 0]
    print(f"GT 集合大小: min={min(sizes)}, max={max(sizes)}, mean={sum(sizes)/len(sizes):.1f}  (若 max>9 说明已按 25 张生成)")
    i0 = next(i for i in range(len(gt_9_list)) if len(gt_9_list[i]) > 0)
    try:
        if i0 < len(gt_25_ordered) and gt_25_ordered[i0] and isinstance(gt_25_ordered[i0][0], (int, np.integer)):
            gt0 = list(gt_25_ordered[i0])
        else:
            gt0 = sorted(gt_9_list[i0])
    except NameError:
        gt0 = sorted(gt_9_list[i0])
    print(f"\n第一张有 GT 的 query（index={i0}）的 25 张 GT 图片:")
    for j, idx in enumerate(gt0):
        idx = int(idx) if not isinstance(idx, (int, np.integer)) else idx
        name = db_names[idx] if idx < len(db_names) else f"<{idx}>"
        print(f"  [{j+1:2d}] {name}")

构建 database_coords（GDAL + id_startx_starty）...
=== database_coords 各场景（未算出坐标的会打印原因）===
  [city1] 共 342 条 -> 有效坐标 342/342
  [city2] 共 168 条 -> 有效坐标 168/168
  [city3] 共 224 条 -> 有效坐标 224/224
  [industry1] 共 868 条 -> 有效坐标 868/868
  [industry2] 共 414 条 -> 有效坐标 414/414
  [industry3] 共 1476 条 -> 有效坐标 1476/1476
  [park1] 共 324 条 -> 有效坐标 324/324
  [rural1] 共 1845 条 -> 有效坐标 1845/1845
  [rural2] 共 1845 条 -> 有效坐标 1845/1845
  [rural3] 共 399 条 -> 有效坐标 399/399
  [school] 共 1178 条 -> 有效坐标 1178/1178
  [suburbs1] 共 480 条 -> 有效坐标 480/480
  [suburbs2] 共 468 条 -> 有效坐标 468/468
  [village1] 共 285 条 -> 有效坐标 285/285
  [village2] 共 693 条 -> 有效坐标 693/693
  -> 合计有效坐标: 11009/11009

有效坐标: 11009/11009
构建坐标法 GT@25（每 query 方圆 25 张）...
=== 坐标法 GT@25 各轨迹（未算出 GT 的会打印原因）===

--- 坐标法 GT 首帧诊断 [city1] ---
大图 mapbox geotransform (6 参数):
  gt[0] 左上角 X (投影/经度): 12123218.434142068
  gt[1] 像元宽:               0.2985821417389691
  gt[2] 旋转(常为 0):         0.0
  gt[3] 左上角 Y (投影/纬度): 4062398.742272254
  gt[4] 旋转(常为 0):         0.0
  

## 3. HNSW 索引（NetVLAD 已归一化，用内积）

In [4]:
K = 20
M = 24
ef_construction = 200
ef_search = 50

# 内积索引（归一化向量上等价于余弦）；Faiss 内积返回越大越相似
index = faiss.IndexHNSWFlat(D, M, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch = ef_search
index.add(db_descs)
print(f"HNSW 索引已构建: M={M}, efConstruction={ef_construction}, efSearch={ef_search}, K={K}")

HNSW 索引已构建: M=24, efConstruction=200, efSearch=50, K=20


## 4. 逐轨迹运行 HNSW + HMM，并计算 Recall@1 / @5 / @10

In [5]:
# 检索方式：按场景检索（每条 query 只在该 query 所属场景的 DB 内检索，非全库检索）
use_coords = n_valid > 0
coords_for_hmm = database_coords if use_coords else None

index_per_scene = {}
for scene_name, db_start, db_end in db_scene_ranges:
    seg = db_descs[db_start:db_end]
    idx = faiss.IndexHNSWFlat(D, M, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = ef_construction
    idx.hnsw.efSearch = ef_search
    idx.add(seg)
    index_per_scene[scene_name] = (idx, db_start, db_end)

pred_hnsw = np.full(n_queries, -1, dtype=np.int64)
pred_hmm = np.full(n_queries, -1, dtype=np.int64)
topk_hnsw = np.full((n_queries, 10), -1, dtype=np.int64)
topk_hmm = np.full((n_queries, 10), -1, dtype=np.int64)
query_idx = 0

for traj_idx, (scene_name, start, end) in enumerate(trajectory_ranges):
    idx_scene, db_start, db_end = index_per_scene.get(scene_name, (None, 0, 0))
    if idx_scene is None:
        query_idx += (end - start)
        continue
    q_descs = query_descs[start:end]
    n_frames = q_descs.shape[0]
    k_scene = min(K, db_end - db_start)
    D_t, I_local = idx_scene.search(q_descs, k_scene)
    I_t = I_local.astype(np.int64) + db_start
    dist_for_hmm = 1.0 - D_t.astype(np.float64)

    # 本轨迹 query 在大图像素坐标，用于 HMM 位移先验（每帧间隔 1/3 秒）
    paths = get_scene_paths(scene_name, DATASETS_BASE)
    lats, lons = load_uav_infos(paths["uav_infos_path"])
    gt_tuple, _ = get_geotransform_and_srs(paths["mapbox_path"])
    n_use = min(len(lats), n_frames)
    query_px = [None] * n_frames
    query_py = [None] * n_frames
    for f in range(n_use):
        px, py = lonlat_to_pixel(lons[f], lats[f], gt_tuple)
        query_px[f], query_py[f] = px, py

    verbose_hmm = (traj_idx == 0)
    if verbose_hmm:
        print("========== 第一个场景 HMM 过程（逐帧）==========")
    hmm = OnlineHMM(coords_for_hmm, K, verbose=verbose_hmm)
    for f in range(n_frames):
        pred_hnsw[query_idx] = I_t[f, 0]
        n_top = min(10, I_t.shape[1])
        topk_hnsw[query_idx, :n_top] = I_t[f, :n_top]
        inds, dists = I_t[f], dist_for_hmm[f]
        if len(inds) < K:
            inds = np.concatenate([inds, np.full(K - len(inds), inds[0], dtype=np.int64)])
            dists = np.concatenate([dists, np.full(K - len(dists), dists[0], dtype=np.float64)])
        else:
            inds, dists = inds[:K], dists[:K]
        displacement = None
        if f >= 1 and query_px[f-1] is not None and query_px[f] is not None:
            displacement = (query_px[f] - query_px[f-1], query_py[f] - query_py[f-1])
        hmm_top = hmm.update(inds, dists, return_top_k=10, displacement=displacement)
        pred_hmm[query_idx] = hmm_top[0]
        for j, idx in enumerate(hmm_top[:10]):
            topk_hmm[query_idx, j] = idx
        # 每场景前 3 帧用 GT 的 top1 做 warm-start，下一帧转移以该 GT 为中心
        if f < 3 and len(gt_9_list[query_idx]) > 0:
            gt_idx = gt_25_ordered[query_idx][0] if (query_idx < len(gt_25_ordered) and gt_25_ordered[query_idx]) else min(gt_9_list[query_idx])
            if verbose_hmm:
                gt_name = db_names[gt_idx] if 0 <= gt_idx < len(db_names) else "?"
                print(f"  [warm-start] 帧 f={f} 用 GT top1 替代: gt_idx={gt_idx} -> {gt_name}")
            pred_hmm[query_idx] = gt_idx
            topk_hmm[query_idx, 0] = gt_idx
            rest = [x for x in hmm_top if x != gt_idx][:9]
            for j, x in enumerate(rest):
                topk_hmm[query_idx, j + 1] = x
            hmm.override_prev_best(gt_idx)
        query_idx += 1

assert query_idx == n_queries

========== 第一个场景 HMM 过程（逐帧）==========

--- HMM Frame 0 ---
  displacement = None
  candidate_indices (HNSW top-K) = [175 174 153 196 195 193 155 217 216 194 177 173 172 192 178 152 151 157
 134 199]
  distances (越小越相似) = [0.78167842 0.79224835 0.7949547  0.79604542 0.80233809 0.8062219
 0.80703425 0.82207556 0.82405195 0.82588693 0.82598315 0.82688189
 0.82956131 0.8302986  0.83211821 0.83570382 0.83578756 0.83914608
 0.8413218  0.84321621]
  log_emission: min=-3.5357, max=-2.3050, argmax=0 -> global_idx=175
  [首帧] 无转移，按 emission 排序 -> top-3 global_idx = [175, 174, 153]
  [warm-start] 帧 f=0 用 GT top1 替代: gt_idx=152

--- HMM Frame 1 ---
  displacement = (-14.913080883758084, 4.51009573291185)
  candidate_indices (HNSW top-K) = [175 174 153 195 193 196 173 172 155 194 192 152 177 216 217 151 178  35
   9 129]
  distances (越小越相似) = [0.7835691  0.7876481  0.79529016 0.80385055 0.80626634 0.81345418
 0.81680198 0.82128552 0.82670626 0.82880293 0.83000404 0.83139732
 0.83180425 0.83242795 0.

In [6]:
valid = np.array([len(gt_9_list[i]) > 0 for i in range(n_queries)])
n_eval = int(np.sum(valid))

# Recall@1 / @5 / @10：top-1、top-5、top-10 中是否有任一张在 GT（方圆 25 张）内；HMM 的 topk 由 update(return_top_k=10) 直接得到
correct_1_hnsw = np.array([topk_hnsw[i, 0] in gt_9_list[i] if topk_hnsw[i, 0] >= 0 else False for i in range(n_queries)])
correct_5_hnsw = np.array([any(topk_hnsw[i, j] in gt_9_list[i] for j in range(5)) for i in range(n_queries)])
correct_10_hnsw = np.array([any(topk_hnsw[i, j] in gt_9_list[i] for j in range(10)) for i in range(n_queries)])
correct_1_hmm = np.array([pred_hmm[i] in gt_9_list[i] for i in range(n_queries)])
correct_5_hmm = np.array([any(topk_hmm[i, j] in gt_9_list[i] for j in range(5)) for i in range(n_queries)])
correct_10_hmm = np.array([any(topk_hmm[i, j] in gt_9_list[i] for j in range(10)) for i in range(n_queries)])

# 诊断：query 与预测分别属于哪个场景（全局检索时预测常落在其它场景导致 Recall=0）
query_to_scene = {}
for scene_name, start, end in trajectory_ranges:
    for i in range(start, end):
        query_to_scene[i] = scene_name
db_idx_to_scene = {}
for scene_name, start, end in db_scene_ranges:
    for j in range(start, end):
        db_idx_to_scene[j] = scene_name
valid_indices = np.where(valid)[0]
n_diag = min(5, len(valid_indices))
print("诊断（前 %d 条有 GT 的 query）：" % n_diag)
for k in range(n_diag):
    i = valid_indices[k]
    q_scene = query_to_scene.get(i, "?")
    p_hnsw = int(pred_hnsw[i])
    p_hmm = int(pred_hmm[i])
    p_hnsw_scene = db_idx_to_scene.get(p_hnsw, "?")
    p_hmm_scene = db_idx_to_scene.get(p_hmm, "?")
    gt_set = (gt_25_ordered[i][:5] if i < len(gt_25_ordered) and gt_25_ordered[i] else list(gt_9_list[i])[:5])
    in_hnsw = correct_1_hnsw[i]
    in_hmm = correct_1_hmm[i]
    print(f"  query {i}: 场景={q_scene} | HNSW pred={p_hnsw} (场景={p_hnsw_scene}) in_GT={in_hnsw} | HMM pred={p_hmm} (场景={p_hmm_scene}) in_GT={in_hmm} | GT 示例={gt_set}")
same_scene_hnsw = sum(1 for i in valid_indices if query_to_scene.get(i) == db_idx_to_scene.get(pred_hnsw[i]))
same_scene_hmm = sum(1 for i in valid_indices if query_to_scene.get(i) == db_idx_to_scene.get(pred_hmm[i]))
print(f"预测与 query 同场景的比例: HNSW {same_scene_hnsw}/{n_eval}, HMM {same_scene_hmm}/{n_eval}")
i0 = valid_indices[0]
p0 = int(pred_hnsw[i0])
gt0 = gt_25_ordered[i0] if (i0 < len(gt_25_ordered) and gt_25_ordered[i0]) else list(gt_9_list[i0])
print("")
print("query 0 细查（看预测与 GT 子图是否相邻）:")
print(f"  预测子图 db_names[{p0}] = {db_names[p0]}")
for g in sorted(gt0)[:5]:
    print(f"  GT 子图 db_names[{g}] = {db_names[g]}")
print("  （若 id_startx_starty 相差很大，说明坐标→瓦片或 query 与 uav_infos 行序可能不一致）")
print("前 3 条 query 的图像名（请与 uav_infos 前 3 行的 file_name 核对是否一一对应）:")
for k in range(min(3, len(query_names))):
    print(f"  query {k}: {query_names[k]}")
print("")

if n_eval > 0:
    r1_hnsw = correct_1_hnsw[valid].mean()
    r5_hnsw = correct_5_hnsw[valid].mean()
    r10_hnsw = correct_10_hnsw[valid].mean()
    r1_hmm = correct_1_hmm[valid].mean()
    r5_hmm = correct_5_hmm[valid].mean()
    r10_hmm = correct_10_hmm[valid].mean()
    print("Recall（仅在有坐标法 GT 的 query 上，命中方圆 25 张即正确）:")
    print(f"  HNSW:        Recall@1={r1_hnsw:.4f}  Recall@5={r5_hnsw:.4f}  Recall@10={r10_hnsw:.4f}  (n={n_eval})")
    print(f"  HNSW + HMM:  Recall@1={r1_hmm:.4f}  Recall@5={r5_hmm:.4f}  Recall@10={r10_hmm:.4f}")
else:
    print("无有效坐标法 GT，请检查 uav_infos.csv 与 GDAL 路径。")

# 将 HNSW 与 HNSW+HMM 的 top-10 结果保存到 ch3/results（每行：查询图像名\t检索到的 db 图像名 x10）
results_dir = Path(_root) / "results"
results_dir.mkdir(parents=True, exist_ok=True)
def _name(i, names): return names[i] if 0 <= i < len(names) else "-"
with open(results_dir / "hnsw_top10.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        db_cols = [_name(int(topk_hnsw[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
with open(results_dir / "hnsw_hmm_top10.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        db_cols = [_name(int(topk_hmm[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
with open(results_dir / "gt_25.txt", "w", encoding="utf-8") as f:
    f.write("query\t" + "\t".join(f"db_{j}" for j in range(1, 26)) + "\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        gt_indices = gt_25_ordered[i][:25] if i < len(gt_25_ordered) and gt_25_ordered[i] else sorted(gt_9_list[i])[:25]
        db_cols = [_name(idx, db_names) for idx in gt_indices]
        db_cols += ["-"] * (25 - len(db_cols))
        f.write("\t".join([q] + db_cols) + "\n")
print(f"按场景检索 Top-10 已保存: {results_dir / 'hnsw_top10.txt'}, {results_dir / 'hnsw_hmm_top10.txt'}")
print(f"GT 已保存: {results_dir / 'gt_25.txt'}")



诊断（前 5 条有 GT 的 query）：
  query 0: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=152 (场景=city1) in_GT=True | GT 示例=[152, 196, 175, 195, 174]
  query 1: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=152 (场景=city1) in_GT=True | GT 示例=[152, 196, 175, 195, 174]
  query 2: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=152 (场景=city1) in_GT=True | GT 示例=[152, 196, 175, 195, 174]
  query 3: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=152 (场景=city1) in_GT=True | GT 示例=[152, 196, 195, 175, 174]
  query 4: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=152 (场景=city1) in_GT=True | GT 示例=[152, 196, 195, 175, 174]
预测与 query 同场景的比例: HNSW 5988/5988, HMM 5988/5988

query 0 细查（看预测与 GT 子图是否相邻）:
  预测子图 db_names[175] = tif/259_1906_2206.tif
  GT 子图 db_names[107] = tif/198_1306_1756.tif
  GT 子图 db_names[108] = tif/199_1456_1756.tif
  GT 子图 db_names[111] = tif/200_1606_1756.tif
  GT 子图 db_names[112] = tif/201_1756_1756.tif
  GT 子图 db_names[113] = ti

## 4b. 全库检索（不按场景）

在**全部 DB 数据**上建一个 HNSW 索引，每条 query 在整库中检索（不限制场景）。

In [7]:
# HMM 使用的坐标（与按场景检索一致；仅跑本 cell 时也需 n_valid、database_coords 已存在）
use_coords = n_valid > 0
coords_for_hmm = database_coords if use_coords else None

# 复用前面已构建的全库 index（见 HNSW 参数 cell），不重复建索引
index_global = index
print("全库检索使用已构建的 index, K=", K)

k_search = min(K, N_db)
D_global, I_global = index_global.search(query_descs, k_search)
I_global = I_global.astype(np.int64)
dist_global = 1.0 - D_global.astype(np.float64)

pred_hnsw_global = np.full(n_queries, -1, dtype=np.int64)
pred_hmm_global = np.full(n_queries, -1, dtype=np.int64)
topk_hnsw_global = np.full((n_queries, 10), -1, dtype=np.int64)
topk_hmm_global = np.full((n_queries, 10), -1, dtype=np.int64)
query_idx = 0
for traj_idx, (scene_name, start, end) in enumerate(trajectory_ranges):
    n_frames = end - start
    paths = get_scene_paths(scene_name, DATASETS_BASE)
    lats, lons = load_uav_infos(paths["uav_infos_path"])
    gt_tuple, _ = get_geotransform_and_srs(paths["mapbox_path"])
    n_use = min(len(lats), n_frames)
    query_px = [None] * n_frames
    query_py = [None] * n_frames
    for f in range(n_use):
        px, py = lonlat_to_pixel(lons[f], lats[f], gt_tuple)
        query_px[f], query_py[f] = px, py
    hmm = OnlineHMM(coords_for_hmm, K)
    for f in range(n_frames):
        pred_hnsw_global[query_idx] = I_global[query_idx, 0]
        n_top = min(10, I_global.shape[1])
        topk_hnsw_global[query_idx, :n_top] = I_global[query_idx, :n_top]
        inds = I_global[query_idx]
        dists = dist_global[query_idx]
        if len(inds) < K:
            inds = np.concatenate([inds, np.full(K - len(inds), inds[0], dtype=np.int64)])
            dists = np.concatenate([dists, np.full(K - len(dists), dists[0], dtype=np.float64)])
        else:
            inds, dists = inds[:K], dists[:K]
        displacement = None
        if f >= 1 and query_px[f-1] is not None and query_px[f] is not None:
            displacement = (query_px[f] - query_px[f-1], query_py[f] - query_py[f-1])
        hmm_top = hmm.update(inds, dists, return_top_k=10, displacement=displacement)
        pred_hmm_global[query_idx] = hmm_top[0]
        for j, idx in enumerate(hmm_top[:10]):
            topk_hmm_global[query_idx, j] = idx
        if f < 3 and len(gt_9_list[query_idx]) > 0:
            gt_idx = gt_25_ordered[query_idx][0] if (query_idx < len(gt_25_ordered) and gt_25_ordered[query_idx]) else min(gt_9_list[query_idx])
            pred_hmm_global[query_idx] = gt_idx
            topk_hmm_global[query_idx, 0] = gt_idx
            rest = [x for x in hmm_top if x != gt_idx][:9]
            for j, x in enumerate(rest):
                topk_hmm_global[query_idx, j + 1] = x
            hmm.override_prev_best(gt_idx)
        query_idx += 1
assert query_idx == n_queries

c1_h = np.array([topk_hnsw_global[i, 0] in gt_9_list[i] if topk_hnsw_global[i, 0] >= 0 else False for i in range(n_queries)])
c5_h = np.array([any(topk_hnsw_global[i, j] in gt_9_list[i] for j in range(5)) for i in range(n_queries)])
c10_h = np.array([any(topk_hnsw_global[i, j] in gt_9_list[i] for j in range(10)) for i in range(n_queries)])
c1_m = np.array([pred_hmm_global[i] in gt_9_list[i] for i in range(n_queries)])
c5_m = np.array([any(topk_hmm_global[i, j] in gt_9_list[i] for j in range(5)) for i in range(n_queries)])
c10_m = np.array([any(topk_hmm_global[i, j] in gt_9_list[i] for j in range(10)) for i in range(n_queries)])
if n_eval > 0:
    print("全库检索 Recall（命中方圆 25 张即正确）:")
    print(f"  HNSW:        Recall@1={c1_h[valid].mean():.4f}  Recall@5={c5_h[valid].mean():.4f}  Recall@10={c10_h[valid].mean():.4f}  (n={n_eval})")
    print(f"  HNSW + HMM:  Recall@1={c1_m[valid].mean():.4f}  Recall@5={c5_m[valid].mean():.4f}  Recall@10={c10_m[valid].mean():.4f}")
else:
    print("无有效 GT，跳过全库 Recall。")

# 全库检索 top-10 保存到 results（格式同按场景：query\tdb_1\t...\tdb_10）
results_dir = Path(_root) / "results"
results_dir.mkdir(parents=True, exist_ok=True)
def _name_global(i, names): return names[i] if 0 <= i < len(names) else "-"
with open(results_dir / "hnsw_top10_global.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name_global(i, query_names)
        db_cols = [_name_global(int(topk_hnsw_global[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
with open(results_dir / "hnsw_hmm_top10_global.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name_global(i, query_names)
        db_cols = [_name_global(int(topk_hmm_global[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
print(f"全库检索 Top-10 已保存: {results_dir / 'hnsw_top10_global.txt'}, {results_dir / 'hnsw_hmm_top10_global.txt'}")



全库检索使用已构建的 index, K= 20


全库检索 Recall（命中方圆 25 张即正确）:
  HNSW:        Recall@1=0.3105  Recall@5=0.5793  Recall@10=0.6660  (n=5988)
  HNSW + HMM:  Recall@1=0.3155  Recall@5=0.5304  Recall@10=0.6528
全库检索 Top-10 已保存: /home/lty/code_my/ch3/results/hnsw_top10_global.txt, /home/lty/code_my/ch3/results/hnsw_hmm_top10_global.txt
